# 11 — Simulation-tuned alpha (CTL-03)

**Decision:** ADR 0060 / CTL-03=B — grid-search the demand fractile `alpha` on closed-loop **episode profit** (SIM-01=B), not the textbook newsvendor ratio `c_s / (c_s + c_w)`.

This notebook uses the production API in `sim/alpha_tune.py`:

- `evaluate_alpha_episode_profit` — score one (arm, alpha) under shared CRN
- `tune_alpha_grid` — pick best alpha on a grid per ladder arm
- `save_tuned_alpha_table` / `load_tuned_alpha_table` — artifact I/O

Arms tuned here: **constant**, **rung0**, **sw**, and **rollout** (damped-SW base + one-step rollout). `dp` remains a hand-filled placeholder (T-031).

Scoring routes through the **Rust `voi_core` kernel** when `blueberries_voi._core.evaluate_alpha_tune_episode_py` is available (after `maturin develop`).

**Defaults are smoke-sized** — short horizon, CI alpha grid, cool shipments. Rollout uses **medium** budgets (H=7, 2 paths). Set `FULL_RUN = True` only when you have time.


## Setup

From the repo root:

```bash
uv sync --extra notebooks --extra viz
uv run maturin develop --manifest-path crates/voi_py/Cargo.toml
uv run jupyter lab
```


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

from blueberries_voi.backend import rust_available, rust_core
from blueberries_voi.sim.alpha_tune import (
    DEFAULT_CI_ALPHAS,
    DEFAULT_CI_CANDIDATE_CASE_RADIUS,
    DEFAULT_CI_N_ROLLOUT_PATHS,
    DEFAULT_CI_ROLLOUT_H,
    DEFAULT_DESKTOP_ALPHAS,
    DEFAULT_TUNED_ALPHA_PATH,
    LADDER_ALPHA_ARMS,
    evaluate_alpha_episode_profit,
    load_tuned_alpha_table,
    save_tuned_alpha_table,
    tune_alpha_grid,
)
from blueberries_voi.sim.bakeoff_rollout import (
    DEFAULT_CANDIDATE_CASE_RADIUS,
    DEFAULT_N_ROLLOUT_PATHS,
    DEFAULT_ROLLOUT_H,
)
from blueberries_voi.sim.profit import DEFAULT_PROFIT_COSTS
from blueberries_voi.sim.shipments import default_shipments, smoke_cool_shipments

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "blueberries_voi").is_dir():
    REPO_ROOT = REPO_ROOT.parent

# --- notebook knobs (keep smoke defaults for interactive runs) ---
FULL_RUN = False
USE_ABDELLA = False  # needs Abdella parquet on disk
ROOT_SEED = 42

if FULL_RUN:
    ALPHAS = DEFAULT_DESKTOP_ALPHAS
    N_BURN, N_SCORE = 14, 28
    ROLLOUT_H = DEFAULT_ROLLOUT_H
    N_ROLLOUT_PATHS = DEFAULT_N_ROLLOUT_PATHS
    CANDIDATE_CASE_RADIUS = DEFAULT_CANDIDATE_CASE_RADIUS
else:
    ALPHAS = tuple(DEFAULT_CI_ALPHAS)
    N_BURN, N_SCORE = 2, 5
    # Medium smoke rollout (interactive but meaningful; slower than SW)
    ROLLOUT_H = 7
    N_ROLLOUT_PATHS = 2
    CANDIDATE_CASE_RADIUS = 1

AVAILABLE_ARMS = ("constant", "rung0", "sw", "rollout")
PLACEHOLDER_ARMS = tuple(a for a in LADDER_ALPHA_ARMS if a not in AVAILABLE_ARMS)

shipments = default_shipments() if USE_ABDELLA else smoke_cool_shipments()
costs = DEFAULT_PROFIT_COSTS
ARTIFACT = REPO_ROOT / "experiments" / "tuned_alpha_notebook.json"


def rollout_tune_kwargs() -> dict[str, int]:
    return {
        "rollout_h": ROLLOUT_H,
        "n_rollout_paths": N_ROLLOUT_PATHS,
        "candidate_case_radius": CANDIDATE_CASE_RADIUS,
    }


def arm_eval_kwargs(arm_id: str) -> dict[str, Any]:
    if arm_id == "rollout":
        return rollout_tune_kwargs()
    return {}


rust_fn = getattr(rust_core, "evaluate_alpha_tune_episode_py", None) if rust_core else None
print(f"Rust kernel: {rust_available() and rust_fn is not None}")
print(f"alpha grid ({len(ALPHAS)}): {ALPHAS}")
print(f"episode: n_burn={N_BURN}, n_score={N_SCORE}")
print(
    f"rollout budgets: H={ROLLOUT_H}, n_paths={N_ROLLOUT_PATHS}, "
    f"radius={CANDIDATE_CASE_RADIUS}"
)
print(f"shipments: {'Abdella' if USE_ABDELLA else 'smoke cool fixture'}")

%matplotlib inline
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})


## Textbook fractile vs simulation tuning (CTL-03)

The single-period newsvendor fractile uses scaffold costs from `DEFAULT_PROFIT_COSTS` (uncalibrated, ADR 0104). CTL-03=B instead maximizes closed-loop episode profit on the grid below.


In [ ]:
alpha_theory_penalty = costs.stockout_penalty / (costs.stockout_penalty + costs.waste_cost)
alpha_theory_margin = costs.unit_margin / (costs.unit_margin + costs.waste_cost)
print(f"Textbook (penalty / waste): {alpha_theory_penalty:.3f}")
print(f"Textbook (margin / waste):  {alpha_theory_margin:.3f}")


## Score one (arm, alpha)

Quick sanity check before running the full grid.


In [ ]:
for demo_arm, demo_alpha in ("sw", 0.9), ("rollout", 0.9):
    p = evaluate_alpha_episode_profit(
        demo_arm,
        demo_alpha,
        ROOT_SEED,
        shipments=shipments,
        costs=costs,
        n_burn=N_BURN,
        n_score=N_SCORE,
        **arm_eval_kwargs(demo_arm),
    )
    print(f"{demo_arm} alpha={demo_alpha}: scored profit = {p:.2f}")


## Profit curves (tqdm over alpha grid)

Shared `ROOT_SEED` gives common random numbers across alpha candidates on the same arm. The rollout curve is the slow cell.


In [ ]:
def profit_curve(arm_id: str) -> tuple[tuple[float, ...], list[float], float]:
    profits: list[float] = []
    kw = arm_eval_kwargs(arm_id)
    for alpha in tqdm(ALPHAS, desc=f"{arm_id} alpha grid"):
        profits.append(
            evaluate_alpha_episode_profit(
                arm_id,
                float(alpha),
                ROOT_SEED,
                shipments=shipments,
                costs=costs,
                n_burn=N_BURN,
                n_score=N_SCORE,
                **kw,
            )
        )
    best_idx = int(np.argmax(profits))
    return ALPHAS, profits, float(ALPHAS[best_idx])


sw_alphas, sw_profits, sw_best = profit_curve("sw")
rollout_alphas, rollout_profits, rollout_best = profit_curve("rollout")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, arm, alphas, profits, best in (
    (axes[0], "sw", sw_alphas, sw_profits, sw_best),
    (axes[1], "rollout", rollout_alphas, rollout_profits, rollout_best),
):
    ax.plot(alphas, profits, "o-", color="#2563eb")
    ax.axvline(best, color="#16a34a", linestyle="--", label=f"alpha* = {best:.2f}")
    ax.axvline(alpha_theory_penalty, color="#dc2626", linestyle=":", label="theory (penalty)")
    ax.set_xlabel("alpha")
    ax.set_title(arm)
    ax.legend(loc="best", fontsize=8)
axes[0].set_ylabel("Episode profit (scored days)")
fig.suptitle("CTL-03 alpha grid search")
fig.tight_layout()
plt.show()

list(zip(sw_alphas, [round(p, 2) for p in sw_profits])),
list(zip(rollout_alphas, [round(p, 2) for p in rollout_profits])),


## Tune all available ladder arms

Includes **rollout** (SW base + one-step rollout). Only `dp` is hand-filled. tqdm wraps the arm loop.


In [ ]:
tuned: dict[str, float] = {}
for arm in tqdm(AVAILABLE_ARMS, desc="tune arms"):
    tuned[arm] = tune_alpha_grid(
        arm,
        alphas=ALPHAS,
        root_seed=ROOT_SEED,
        shipments=shipments,
        costs=costs,
        n_burn=N_BURN,
        n_score=N_SCORE,
        **arm_eval_kwargs(arm),
    )

for arm in PLACEHOLDER_ARMS:
    tuned[arm] = 0.9  # placeholder until T-031 (toy DP)

tuned


## Save tuned alpha artifact

Writes JSON under `experiments/` (default production path is `experiments/tuned_alpha.json`). This notebook uses a notebook-specific filename.


In [ ]:
header = {
    "notebook": "11_simulation_alpha_tuning",
    "full_run": FULL_RUN,
    "n_burn": N_BURN,
    "n_score": N_SCORE,
    "alphas": list(ALPHAS),
    "root_seed": ROOT_SEED,
    "rust_kernel": bool(rust_available() and rust_fn is not None),
    "rollout_h": ROLLOUT_H,
    "n_rollout_paths": N_ROLLOUT_PATHS,
    "candidate_case_radius": CANDIDATE_CASE_RADIUS,
}
save_tuned_alpha_table(
    ARTIFACT,
    tuned,
    header=header,
)
loaded = load_tuned_alpha_table(ARTIFACT)
print(f"Saved {ARTIFACT}")
print(loaded)


## Takeaways

1. **CTL-03=B** picks alpha by simulation profit, not the textbook newsvendor fractile.
2. **`tune_alpha_grid`** runs per ladder arm under a shared `root_seed` (CRN across alpha candidates).
3. **Rollout arm** tunes alpha on the damped-SW base inside one-step rollout — pass `rollout_h`, `n_rollout_paths`, and `candidate_case_radius`.
4. **Rust path** — when `_core` is built, scoring uses f-native `voi_core` physics; rebuild with `maturin develop` after Rust changes.
5. **Scale up** — set `FULL_RUN = True` for desktop alpha grid, longer episodes, and full rollout budgets (H=28, 8 paths).
